This notebook was created with the intention of constructing a high accuracy model (instead of a balanced approach)

`The following cell contains: experiment configuration, random seeds, device, data directories, output directory, image configuration, training configuration and experiment configuration`

In [1]:
from pathlib import Path
import os
import random
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path(
    r"C:\Users\Pichau\Desktop\python intenso\IC-DeepLearning"
)

DATA_ROOT = Path(r"D:\IC-DeepLearning")

ORGANIZING_DIR = PROJECT_ROOT / "data_organizing"
MODELS_DIR = PROJECT_ROOT / "sunspot_prediction_models"

METADATA_CSV = ORGANIZING_DIR / "final_output_stream_mcintosh_split_mag.csv"

IC_CROPS_FOLDER = DATA_ROOT / "cropped_data_v2"
MAG_CROPS_FOLDER = DATA_ROOT / "cropped_data_v2_mag"

OUTPUT_DIR = DATA_ROOT / "results" / "accuracy_temporal_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 2024, 7, 99]

IMAGE_SIZE = 101
BATCH_SIZE = 32
MAX_EPOCHS = 60
LEARNING_RATE = 0.0005
WEIGHT_DECAY = 1e-4
PATIENCE = 10

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

required_paths = [
    METADATA_CSV,
    IC_CROPS_FOLDER,
    MAG_CROPS_FOLDER,
]

for path in required_paths:
    assert path.exists(), f"Path not found: {path}"

print("Configuration loaded successfully!")
print("-" * 50)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Metadata:", METADATA_CSV)
print("Continuum:", IC_CROPS_FOLDER)
print("Magnetograms:", MAG_CROPS_FOLDER)
print("Output:", OUTPUT_DIR)
print("Seeds:", SEEDS)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", MAX_EPOCHS)
print("Learning rate:", LEARNING_RATE)


Configuration loaded successfully!
--------------------------------------------------
Device: cuda
GPU: NVIDIA GeForce RTX 2060 SUPER
Metadata: C:\Users\Pichau\Desktop\python intenso\IC-DeepLearning\data_organizing\final_output_stream_mcintosh_split_mag.csv
Continuum: D:\IC-DeepLearning\cropped_data_v2
Magnetograms: D:\IC-DeepLearning\cropped_data_v2_mag
Output: D:\IC-DeepLearning\results\accuracy_temporal_v1
Seeds: [42, 123, 2024, 7, 99]
Image size: 101
Batch size: 32
Maximum epochs: 60
Learning rate: 0.0005


In [2]:
df = pd.read_csv(METADATA_CSV)

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["year"] = df["date"].dt.year

class_col = (
    "zurich_class" if "zurich_class" in df.columns else "z_class"
)

print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)

print("Total observations:", len(df))
print("Available columns:", df.columns.tolist())
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Invalid dates:", df["date"].isna().sum())

# Per year
print("\nOBSERVATIONS PER YEAR")
print(df.groupby("year").size().to_string())

# Class distribution
print("\nOVERALL CLASS DISTRIBUTION")
print(df[class_col].value_counts(dropna=False).sort_index())

# Class distribution by year
print("\nCLASS DISTRIBUTION BY YEAR")
year_class_counts = pd.crosstab(df["year"], df[class_col])
print(year_class_counts.to_string())

# Active-regions per year
if "nmbr" in df.columns:
    print("\nUNIQUE NOAA NUMBERS PER YEAR")
    print(df.groupby("year")["nmbr"].nunique().to_string())

# Check paired crop
if {"crop_filename", "mag_crop_filename"}.issubset(df.columns):
    valid_pairs = (
        df["crop_filename"].notna()
        & df["mag_crop_filename"].notna()
    )
    print("\nPAIRED CROP FILENAMES")
    print("Rows with both filenames:", valid_pairs.sum())
    print("Rows without a complete pair:", (~valid_pairs).sum())

# Check missing values in relevant columns
relevant_cols = [
    col for col in [
        "date", "nmbr", class_col, "crop_filename",
        "mag_crop_filename", "lon"
    ] if col in df.columns
]

print("\nMISSING VALUES IN RELEVANT COLUMNS")
print(df[relevant_cols].isna().sum().to_string())

print("\nAudit completed.")


DATASET OVERVIEW
Total observations: 11518
Available columns: ['nmbr', 'location', 'lat', 'lon', 'carrington_lo', 'area', 'z_class', 'll', 'nn', 'mag_type', 'date', 'pixel_x', 'pixel_y', 'crop_filename', 'mag_class', 'zurich_class', 'penumbra_class', 'compactness_class', 'mag_crop_filename', 'mag_file', 'mag_offset_min', 'rolled_180', 'nan_mismatch', 'year']
Date range: 2002-01-01 00:00:00 to 2011-12-31 00:00:00
Invalid dates: 0

OBSERVATIONS PER YEAR
year
2002    3528
2003    2146
2004    1309
2005     973
2006     600
2007     308
2008     122
2009     123
2010     622
2011    1787

OVERALL CLASS DISTRIBUTION
zurich_class
A    1044
B    1136
C    2341
D    3065
E    1125
F     341
H    2466
Name: count, dtype: int64

CLASS DISTRIBUTION BY YEAR
zurich_class    A    B    C     D    E    F    H
year                                            
2002          268  258  717  1006  380  138  761
2003          171  161  479   647  185   68  435
2004          106  122  230   322  164   63  302

In [3]:
# Audit image availability by year and Zurich class

has_continuum = df["crop_filename"].notna()
has_magnetogram = df["mag_crop_filename"].notna()

df["data_type"] = "Neither image"
df.loc[has_continuum, "data_type"] = "Continuum only"
df.loc[has_magnetogram, "data_type"] = "Magnetogram only"
df.loc[has_continuum & has_magnetogram, "data_type"] = "Complete pair"

print("=" * 60)
print("IMAGE AVAILABILITY")
print("=" * 60)

print("\nAvailability combinations:")
print(pd.crosstab(has_continuum, has_magnetogram).to_string())

print("\nObservations by data type:")
print(df["data_type"].value_counts().to_string())

# Complete pairs by year
paired_df = df[has_continuum & has_magnetogram].copy()

print("\nCOMPLETE PAIRS BY YEAR")
print(paired_df.groupby("year").size().to_string())

print("\nCOMPLETE PAIRS BY ZURICH CLASS")
print(
    paired_df["zurich_class"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nCOMPLETE PAIRS: YEAR x CLASS")
paired_year_class = pd.crosstab(
    paired_df["year"],
    paired_df["zurich_class"]
)
print(paired_year_class.to_string())

# Continuum availability by year
continuum_df = df[has_continuum].copy()

print("\nCONTINUUM OBSERVATIONS BY YEAR")
print(continuum_df.groupby("year").size().to_string())

print("\nCONTINUUM: YEAR x CLASS")
print(
    pd.crosstab(
        continuum_df["year"],
        continuum_df["zurich_class"]
    ).to_string()
)


IMAGE AVAILABILITY

Availability combinations:
mag_crop_filename  False  True 
crop_filename                  
False               2364      0
True                2350   6804

Observations by data type:
data_type
Complete pair     6804
Neither image     2364
Continuum only    2350

COMPLETE PAIRS BY YEAR
year
2002    2565
2003    1400
2004     785
2005     741
2006     479
2007     232
2008      98
2009      97
2010     407

COMPLETE PAIRS BY ZURICH CLASS
zurich_class
A     612
B     649
C    1368
D    1841
E     681
F     215
H    1438

COMPLETE PAIRS: YEAR x CLASS
zurich_class    A    B    C    D    E    F    H
year                                           
2002          195  200  506  722  298  115  529
2003          107  119  303  431  132   40  268
2004           66   74  144  195   94   37  175
2005           45   42  128  251   77   14  184
2006           66   38  139   99   38    0   99
2007           27   32   45   46   12    0   70
2008           18   25   19   21    2    0 

In [4]:
df_cnn = df[df["year"] <= 2010].copy()

# Main dataset for continuum + magnetogram comparison
df_paired = df_cnn[
    df_cnn["crop_filename"].notna()
    & df_cnn["mag_crop_filename"].notna()
].copy()

# Dataset available for continuum-only
df_continuum = df_cnn[
    df_cnn["crop_filename"].notna()
].copy()

print("CNN dataset period:", df_cnn["year"].min(), "-", df_cnn["year"].max())
print("All observations:", len(df_cnn))
print("Continuum observations:", len(df_continuum))
print("Complete image pairs:", len(df_paired))

CNN dataset period: 2002 - 2010
All observations: 9731
Continuum observations: 9154
Complete image pairs: 6804


In [5]:
# Analyze active-region distribution in the paired dataset

region_summary = (
    df_paired.groupby(["year", "nmbr"])
    .agg(
        observations=("zurich_class", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        classes=(
            "zurich_class",
            lambda x: ", ".join(
                sorted(x.dropna().astype(str).unique())
            )
        )
    )
    .reset_index()
)

print("=" * 60)
print("ACTIVE REGION DISTRIBUTION")
print("=" * 60)

print("\nUnique NOAA-number/year groups:")
print(len(region_summary))

print("\nGroups by year:")
print(region_summary.groupby("year").size().to_string())

print("\nObservations per region summary:")
print(region_summary["observations"].describe().to_string())

print("\nLargest region groups:")
print(
    region_summary
    .sort_values("observations", ascending=False)
    .head(15)
    .to_string(index=False)
)

print("\nMissing NOAA numbers:")
print(df_paired["nmbr"].isna().sum())

ACTIVE REGION DISTRIBUTION

Unique NOAA-number/year groups:
1177

Groups by year:
year
2002    416
2003    247
2004    147
2005    111
2006     82
2007     40
2008     24
2009     29
2010     81

Observations per region summary:
count    1177.000000
mean        5.780799
std         3.688953
min         1.000000
25%         3.000000
50%         5.000000
75%         8.000000
max        14.000000

Largest region groups:
 year  nmbr  observations first_date  last_date    classes
 2002    69            14 2002-08-12 2002-08-25    C, E, H
 2002    67            14 2002-08-11 2002-08-24 C, D, E, H
 2002    63            14 2002-08-07 2002-08-20    D, E, H
 2002    61            14 2002-08-04 2002-08-17 C, D, E, H
 2002    30            14 2002-07-10 2002-07-23    C, E, F
 2003   325            14 2003-03-28 2003-04-10 C, D, E, H
 2003   346            14 2003-04-25 2003-05-08       C, H
 2003   337            14 2003-04-19 2003-05-02 C, D, E, H
 2004   606            14 2004-05-08 2004-05-21 

In [6]:
# Inspect observations around year boundaries

df_boundary = df_paired.copy()
df_boundary["date"] = pd.to_datetime(df_boundary["date"])

print("=" * 70)
print("OBSERVATIONS AROUND YEAR BOUNDARIES")
print("=" * 70)

for year in range(2002, 2010):

    start_date = pd.Timestamp(f"{year}-12-15")
    end_date = pd.Timestamp(f"{year + 1}-01-15")

    boundary_data = df_boundary[
        (df_boundary["date"] >= start_date)
        & (df_boundary["date"] <= end_date)
    ].copy()

    print(f"\n--- {year} / {year + 1} ---")

    if boundary_data.empty:
        print("No observations in this interval.")
        continue

    print(
        boundary_data[
            [
                "date",
                "year",
                "nmbr",
                "lat",
                "lon",
                "carrington_lo",
                "zurich_class"
            ]
        ]
        .sort_values("date")
        .to_string(index=False)
    )

OBSERVATIONS AROUND YEAR BOUNDARIES

--- 2002 / 2003 ---
      date  year  nmbr  lat  lon  carrington_lo zurich_class
2002-12-15  2002   215  -18   48            213            A
2002-12-15  2002   218  -18   11            176            D
2002-12-15  2002   220  -13   18            183            E
2002-12-15  2002   221   20    1            166            C
2002-12-15  2002   223   23  -46            119            D
2002-12-15  2002   224  -18  -49            116            E
2002-12-15  2002   225   17  -51            114            D
2002-12-15  2002   226  -28  -39            126            D
2002-12-15  2002   227    6  -21            144            C
2002-12-15  2002   228  -14  -65            100            H
2002-12-15  2002   229   15  -63            102            D
2002-12-16  2002   229   17  -51            101            D
2002-12-16  2002   228  -15  -52            100            H
2002-12-16  2002   227    6   -8            144            D
2002-12-16  2002   226  -28 

In [7]:
# Detect NOAA groups continuing across calendar years

boundary_matches = []

for year in range(2002, 2010):

    december = df_boundary[
        (df_boundary["date"] >= pd.Timestamp(f"{year}-12-25"))
        & (df_boundary["date"] <= pd.Timestamp(f"{year}-12-31"))
    ]

    january = df_boundary[
        (df_boundary["date"] >= pd.Timestamp(f"{year + 1}-01-01"))
        & (df_boundary["date"] <= pd.Timestamp(f"{year + 1}-01-07"))
    ]

    common_numbers = set(december["nmbr"]) & set(january["nmbr"])

    for number in sorted(common_numbers):

        dec_group = december[december["nmbr"] == number]
        jan_group = january[january["nmbr"] == number]

        last_dec = dec_group.sort_values("date").iloc[-1]
        first_jan = jan_group.sort_values("date").iloc[0]

        lat_difference = abs(
            float(last_dec["lat"]) - float(first_jan["lat"])
        )

        carrington_difference = abs(
            float(last_dec["carrington_lo"])
            - float(first_jan["carrington_lo"])
        )

        carrington_difference = min(
            carrington_difference,
            360 - carrington_difference
        )

        boundary_matches.append({
            "year_transition": f"{year}/{year + 1}",
            "nmbr": number,
            "last_december_date": last_dec["date"],
            "first_january_date": first_jan["date"],
            "gap_days": (
                first_jan["date"] - last_dec["date"]
            ).days,
            "latitude_difference": lat_difference,
            "carrington_difference": carrington_difference
        })

boundary_matches_df = pd.DataFrame(boundary_matches)

print("=" * 70)
print("POTENTIAL CONTINUOUS ACTIVE REGIONS")
print("=" * 70)

if boundary_matches_df.empty:
    print("No matching NOAA groups found.")
else:
    print(boundary_matches_df.to_string(index=False))

    print("\nTotal potential continuities:")
    print(len(boundary_matches_df))

POTENTIAL CONTINUOUS ACTIVE REGIONS
year_transition  nmbr last_december_date first_january_date  gap_days  latitude_difference  carrington_difference
      2002/2003   234         2002-12-31         2003-01-01         1                  2.0                    0.0
      2002/2003   239         2002-12-31         2003-01-01         1                  0.0                    1.0
      2005/2006   838         2005-12-31         2006-01-01         1                  0.0                    2.0
      2005/2006   840         2005-12-31         2006-01-01         1                  1.0                    0.0
      2005/2006   841         2005-12-31         2006-01-01         1                  0.0                    4.0
      2005/2006   843         2005-12-31         2006-01-01         1                  0.0                    0.0
      2005/2006   844         2005-12-31         2006-01-01         1                  0.0                    2.0
      2006/2007   933         2006-12-31         200

In [8]:
# Inspect the nine potential cross-year continuities

transitions = [
    (2002, 234),
    (2002, 239),
    (2005, 838),
    (2005, 840),
    (2005, 841),
    (2005, 843),
    (2005, 844),
    (2006, 933),
    (2009, 1039)
]

boundary_check = df_paired.copy()
boundary_check["date"] = pd.to_datetime(boundary_check["date"])

columns = [
    "date",
    "year",
    "nmbr",
    "lat",
    "lon",
    "carrington_lo",
    "zurich_class",
    "location"
]

for year, number in transitions:

    start = pd.Timestamp(f"{year}-12-28")
    end = pd.Timestamp(f"{year + 1}-01-05")

    selected = boundary_check[
        (boundary_check["nmbr"] == number)
        & (boundary_check["date"] >= start)
        & (boundary_check["date"] <= end)
    ]

    print("\n" + "=" * 70)
    print(f"REGION {number}: {year} / {year + 1}")
    print("=" * 70)

    print(
        selected[columns]
        .sort_values("date")
        .to_string(index=False)
    )


REGION 234: 2002 / 2003
      date  year  nmbr  lat  lon  carrington_lo zurich_class location
2002-12-28  2002   234   18  -32            322            H   N18E32
2002-12-29  2002   234   19  -19            322            C   N19E19
2002-12-30  2002   234   18   -5            322            H   N18E05
2002-12-31  2002   234   18    8            322            H   N18W08
2003-01-01  2003   234   20   21            322            H   N20W21
2003-01-02  2003   234   19   33            321            H   N19W33
2003-01-03  2003   234   19   47            322            H   N19W47
2003-01-04  2003   234   19   60            322            A   N19W60

REGION 239: 2002 / 2003
      date  year  nmbr  lat  lon  carrington_lo zurich_class location
2002-12-30  2002   239  -10  -37            290            C   S10E37
2002-12-31  2002   239  -10  -24            290            D   S10E24
2003-01-01  2003   239  -10  -10            291            D   S10E10
2003-01-02  2003   239   -9    4        

In [9]:
# Create region groups that preserve cross-year continuity

df_paired = df_paired.copy()

# Initial identifier: NOAA number + year
df_paired["region_group_id"] = (
    df_paired["year"].astype(str)
    + "_"
    + df_paired["nmbr"].astype(str)
)

# Cross-year continuities identified in the audit
continuities = [
    ((2002, 234), (2003, 234)),
    ((2002, 239), (2003, 239)),
    ((2005, 838), (2006, 838)),
    ((2005, 840), (2006, 840)),
    ((2005, 841), (2006, 841)),
    ((2005, 843), (2006, 843)),
    ((2005, 844), (2006, 844)),
    ((2006, 933), (2007, 933)),
    ((2009, 1039), (2010, 1039))
]

# Merge identifiers belonging to the same continuous region
for (year_a, number_a), (year_b, number_b) in continuities:

    id_a = f"{year_a}_{number_a}"
    id_b = f"{year_b}_{number_b}"

    df_paired["region_group_id"] = (
        df_paired["region_group_id"]
        .replace(id_b, id_a)
    )

print("Original NOAA/year groups:", 1177)
print(
    "Corrected region groups:",
    df_paired["region_group_id"].nunique()
)

print("\nObservations per corrected group:")
print(
    df_paired.groupby("region_group_id")
    .size()
    .describe()
)

print("\nGroups with more than 14 observations:")
print(
    df_paired.groupby("region_group_id")
    .size()
    .loc[lambda x: x > 14]
    .sort_values(ascending=False)
)

Original NOAA/year groups: 1177
Corrected region groups: 1168

Observations per corrected group:
count    1168.000000
mean        5.825342
std         3.711809
min         1.000000
25%         3.000000
50%         5.000000
75%         8.000000
max        14.000000
dtype: float64

Groups with more than 14 observations:
Series([], dtype: int64)


In [10]:
# Analyze the distribution of observations and active regions
# across years and Zurich classes

assert "region_group_id" in df_paired.columns

print("=" * 65)
print("1. OBSERVATIONS BY YEAR AND ZURICH CLASS")
print("=" * 65)

observations_table = pd.crosstab(
    df_paired["year"],
    df_paired["zurich_class"]
)

print(observations_table.to_string())

print("\n" + "=" * 65)
print("2. UNIQUE ACTIVE REGIONS BY YEAR AND ZURICH CLASS")
print("=" * 65)

regions_table = (
    df_paired
    .groupby(["year", "zurich_class"])["region_group_id"]
    .nunique()
    .unstack(fill_value=0)
)

print(regions_table.to_string())

print("\n" + "=" * 65)
print("3. TOTAL UNIQUE REGIONS BY YEAR")
print("=" * 65)

regions_per_year = (
    df_paired.groupby("year")["region_group_id"]
    .nunique()
)

print(regions_per_year.to_string())

print("\nTotal unique active regions:")
print(df_paired["region_group_id"].nunique())

1. OBSERVATIONS BY YEAR AND ZURICH CLASS
zurich_class    A    B    C    D    E    F    H
year                                           
2002          195  200  506  722  298  115  529
2003          107  119  303  431  132   40  268
2004           66   74  144  195   94   37  175
2005           45   42  128  251   77   14  184
2006           66   38  139   99   38    0   99
2007           27   32   45   46   12    0   70
2008           18   25   19   21    2    0   13
2009           19   41    8   17    8    0    4
2010           69   78   76   59   20    9   96

2. UNIQUE ACTIVE REGIONS BY YEAR AND ZURICH CLASS
zurich_class    A    B    C    D   E   F    H
year                                         
2002          141  132  247  212  76  29  196
2003           75   81  148  136  33  11  109
2004           47   48   81   78  31  10   69
2005           33   32   63   72  23   4   69
2006           41   27   59   38  11   0   42
2007           17   19   23   17   3   0   17
2008        

In [11]:
# Analyze Zurich class diversity within each active region

region_class_summary = (
    df_paired.groupby("region_group_id")
    .agg(
        observations=("zurich_class", "size"),
        unique_classes=("zurich_class", "nunique"),
        classes=(
            "zurich_class",
            lambda x: ", ".join(
                sorted(x.dropna().astype(str).unique())
            )
        ),
        first_date=("date", "min"),
        last_date=("date", "max")
    )
    .reset_index()
)

print("=" * 65)
print("CLASS DIVERSITY PER ACTIVE REGION")
print("=" * 65)

print("\nNumber of regions by number of distinct classes:")

print(
    region_class_summary["unique_classes"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nPercentage of regions by class diversity:")

print(
    (
        region_class_summary["unique_classes"]
        .value_counts(normalize=True)
        .sort_index() * 100
    ).round(2).to_string()
)

print("\nRegions with the greatest class diversity:")

print(
    region_class_summary
    .sort_values(
        ["unique_classes", "observations"],
        ascending=False
    )
    .head(15)
    .to_string(index=False)
)

CLASS DIVERSITY PER ACTIVE REGION

Number of regions by number of distinct classes:
unique_classes
1    269
2    349
3    326
4    176
5     44
6      4

Percentage of regions by class diversity:
unique_classes
1    23.03
2    29.88
3    27.91
4    15.07
5     3.77
6     0.34

Regions with the greatest class diversity:
region_group_id  observations  unique_classes          classes first_date  last_date
      2002_9878            13               6 B, C, D, E, F, H 2002-03-21 2002-04-02
       2003_348            13               6 A, B, C, D, F, H 2003-04-27 2003-05-09
      2002_9940            11               6 A, B, C, D, E, H 2002-05-05 2002-05-15
      2010_1054            11               6 A, B, C, D, E, H 2010-03-11 2010-03-21
      2002_9876            13               5    A, C, D, E, H 2002-03-21 2002-04-02
      2002_9919            13               5    C, D, E, F, H 2002-04-23 2002-05-05
       2004_554            13               5    A, C, D, E, H 2004-02-08 2004-02-20

In [ ]:
# Analyze global class distribution using observations and unique active-region groups

print("=" * 65)
print("GLOBAL CLASS DISTRIBUTION")
print("=" * 65)

# Number of image observations per class
class_observations = (
    df_paired["zurich_class"]
    .value_counts()
    .sort_index()
)

# Number of distinct active-region groups per class
class_regions = (
    df_paired.groupby("zurich_class")["region_group_id"]
    .nunique()
    .sort_index()
)

# Combine the results
class_distribution = pd.DataFrame({
    "observations": class_observations,
    "unique_regions": class_regions
})

class_distribution["observation_percentage"] = (
    class_distribution["observations"]
    / class_distribution["observations"].sum()
    * 100
)

class_distribution["region_percentage"] = (
    class_distribution["unique_regions"]
    / class_distribution["unique_regions"].sum()
    * 100
)

print("\nCounts and percentages:")
print(class_distribution.round(2).to_string())

print("\nTotal observations:")
print(class_distribution["observations"].sum())

print("\nTotal unique regions:")
print(df_paired["region_group_id"].nunique())

GLOBAL CLASS DISTRIBUTION

Counts and percentages:
              observations  unique_regions  observation_percentage  region_percentage
zurich_class                                                                         
A                      612             423                    8.99              14.62
B                      649             415                    9.54              14.34
C                     1368             680                   20.11              23.51
D                     1841             593                   27.06              20.50
E                      681             192                   10.01               6.64
F                      215              57                    3.16               1.97
H                     1438             533                   21.13              18.42

Total observations:
6804

Total unique regions:
1168


In [13]:
# Average number of observations per active region for each class

class_distribution["observations_per_region"] = (
    class_distribution["observations"]
    / class_distribution["unique_regions"]
)

print("\nAverage observations per region for each class:")
print(
    class_distribution[
        ["observations", "unique_regions", "observations_per_region"]
    ].round(2).to_string()
)


Average observations per region for each class:
              observations  unique_regions  observations_per_region
zurich_class                                                       
A                      612             423                     1.45
B                      649             415                     1.56
C                     1368             680                     2.01
D                     1841             593                     3.10
E                      681             192                     3.55
F                      215              57                     3.77
H                     1438             533                     2.70


In [14]:
# Analyze the number of observations and active-region groups per year

year_distribution = (
    df_paired.groupby("year")
    .agg(
        observations=("zurich_class", "size"),
        unique_regions=("region_group_id", "nunique")
    )
    .sort_index()
)

print("=" * 65)
print("TEMPORAL DISTRIBUTION OF THE PAIRED DATASET")
print("=" * 65)

print(year_distribution.to_string())

print("\nTotal observations:", df_paired.shape[0])
print("Total unique active regions:", df_paired["region_group_id"].nunique())

TEMPORAL DISTRIBUTION OF THE PAIRED DATASET
      observations  unique_regions
year                              
2002          2565             416
2003          1400             247
2004           785             147
2005           741             111
2006           479              82
2007           232              40
2008            98              24
2009            97              29
2010           407              81

Total observations: 6804
Total unique active regions: 1168


In [15]:
# Analyze Zurich class distribution across years

observations_by_year_class = pd.crosstab(
    df_paired["year"],
    df_paired["zurich_class"]
)

regions_by_year_class = (
    df_paired.groupby(["year", "zurich_class"])["region_group_id"]
    .nunique()
    .unstack(fill_value=0)
)

print("=" * 65)
print("OBSERVATIONS PER YEAR AND ZURICH CLASS")
print("=" * 65)

print(observations_by_year_class.to_string())

print("\n" + "=" * 65)
print("UNIQUE REGIONS PER YEAR AND ZURICH CLASS")
print("=" * 65)

print(regions_by_year_class.to_string())

OBSERVATIONS PER YEAR AND ZURICH CLASS
zurich_class    A    B    C    D    E    F    H
year                                           
2002          195  200  506  722  298  115  529
2003          107  119  303  431  132   40  268
2004           66   74  144  195   94   37  175
2005           45   42  128  251   77   14  184
2006           66   38  139   99   38    0   99
2007           27   32   45   46   12    0   70
2008           18   25   19   21    2    0   13
2009           19   41    8   17    8    0    4
2010           69   78   76   59   20    9   96

UNIQUE REGIONS PER YEAR AND ZURICH CLASS
zurich_class    A    B    C    D   E   F    H
year                                         
2002          141  132  247  212  76  29  196
2003           75   81  148  136  33  11  109
2004           47   48   81   78  31  10   69
2005           33   32   63   72  23   4   69
2006           41   27   59   38  11   0   42
2007           17   19   23   17   3   0   17
2008           15   13 

In [16]:
# Summarize class coverage and dataset size for each year

year_coverage = pd.DataFrame(index=regions_by_year_class.index)

year_coverage["observations"] = year_distribution["observations"]
year_coverage["unique_regions"] = year_distribution["unique_regions"]

year_coverage["classes_present"] = (
    (regions_by_year_class > 0).sum(axis=1)
)

year_coverage["class_F_regions"] = regions_by_year_class["F"]

year_coverage["all_classes_present"] = (
    year_coverage["classes_present"] == len(regions_by_year_class.columns)
)

print("=" * 65)
print("YEARLY CLASS COVERAGE")
print("=" * 65)

print(year_coverage.to_string())

YEARLY CLASS COVERAGE
      observations  unique_regions  classes_present  class_F_regions  all_classes_present
year                                                                                     
2002          2565             416                7               29                 True
2003          1400             247                7               11                 True
2004           785             147                7               10                 True
2005           741             111                7                4                 True
2006           479              82                6                0                False
2007           232              40                6                0                False
2008            98              24                6                0                False
2009            97              29                6                0                False
2010           407              81                7                3          

In [17]:
# Compare class distribution in the original dataset
# and in the paired dataset, restricted to 2002-2010

df_original_period = df[df["year"].between(2002, 2010)].copy()

original_counts = (
    df_original_period["zurich_class"]
    .value_counts()
    .sort_index()
)

paired_counts = (
    df_paired["zurich_class"]
    .value_counts()
    .sort_index()
)

availability_comparison = pd.DataFrame({
    "original_records": original_counts,
    "paired_records": paired_counts
}).fillna(0).astype(int)

availability_comparison["retained_percentage"] = (
    availability_comparison["paired_records"]
    / availability_comparison["original_records"]
    * 100
)

print("=" * 65)
print("ORIGINAL VS PAIRED DATASET")
print("=" * 65)

print(availability_comparison.round(2).to_string())

ORIGINAL VS PAIRED DATASET
              original_records  paired_records  retained_percentage
zurich_class                                                       
A                          883             612                69.31
B                          882             649                73.58
C                         1980            1368                69.09
D                         2636            1841                69.84
E                          944             681                72.14
F                          312             215                68.91
H                         2094            1438                68.67


In [18]:
# Compare original and paired records across years

original_by_year = (
    df_original_period.groupby("year")
    .size()
    .rename("original_records")
)

paired_by_year = (
    df_paired.groupby("year")
    .size()
    .rename("paired_records")
)

year_retention = pd.concat(
    [original_by_year, paired_by_year],
    axis=1
)

year_retention["retained_percentage"] = (
    year_retention["paired_records"]
    / year_retention["original_records"]
    * 100
)

print("=" * 65)
print("YEARLY ORIGINAL VS PAIRED DATASET")
print("=" * 65)

print(year_retention.round(2).to_string())

YEARLY ORIGINAL VS PAIRED DATASET
      original_records  paired_records  retained_percentage
year                                                       
2002              3528            2565                72.70
2003              2146            1400                65.24
2004              1309             785                59.97
2005               973             741                76.16
2006               600             479                79.83
2007               308             232                75.32
2008               122              98                80.33
2009               123              97                78.86
2010               622             407                65.43
